# Titanic Top 4% with ensemble modeling

**Yassine Ghouzam, PhD**

13/07/2017

* **1 Introduction**
* **2 Load and check data**
  * 2.1 load data
  * 2.2 Outlier detection
  * 2.3 joining train and test set
  * 2.4 check for null and missing values
* **3 Feature analysis**
  * 3.1 Numerical values
  * 3.2 Categorical values
* **4 Filling missing Values**
  * 4.1 Age
* **5 Feature engineering**
  * 5.1 Name/Title
  * 5.2 Family Size
  * 5.3 Cabin
  * 5.4 Ticket
* **6 Modeling**
  * 6.1 Simple modeling
    * 6.1.1 Cross validate models
    * 6.1.2 Hyperparameter tuning for best models
    * 6.1.3 Plot learning curves
    * 6.1.4 Feature importance of the tree based classifiers
  * 6.2 Ensemble modeling
    * 6.2.1 Combining models
  * 6.3 Prediction
    * 6.3.1 Predict and Submit results

# 1. Introduction

This is my first kernel at Kaggle. I choosed the Titanic competition which is a good way to introduce feature engineering and ensemble modeling. Firstly, I will display some feature analyses then I'll focus on the feature engineering. Last part concerns modeling and predicting the survival on the Titanic using an voting procedure.

This script follows three main parts:

* Feature analysis
* Feature engineering
* Modeling

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from collections import Counter

from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold, learning_curve

sns.set(style='white', context='notebook', palette='deep')

# 2. Load and check data

## 2.1 Load data

In [2]:
train = pd.read_csv('./input/train.csv')
test = pd.read_csv('./input/test.csv')
IDtest = test['PassengerId']

## 2.2 Outlier detection

In [3]:
def detect_outliers(df, n, features):
    '''
    Takes a dataframe df of features and returns a list of the indices
    corresponding to the observations containing more than n outliers according
    to the Tukey method.
    '''
    outlier_indices = []

    for col in features:
        Q1 = np.percentile(df[col], 25)
        Q3 = np.percentile(df[col], 75)
        IQR = Q3 - Q1

        outlier_step = 1.5 * IQR

        outlier_list_col = df[(df[col] < Q1 - outlier_step) | (df[col] > Q3 + outlier_step)].index

        outlier_indices.extend(outlier_list_col)
    
    outlier_indices = Counter(outlier_indices)
    multiple_outliers = list(k for k, v in outlier_indices.items() if v > n)

    return multiple_outliers

Outliers_to_drop = detect_outliers(train, 2, ['Age', 'SibSp', 'Parch', 'Fare'])

Since outliers can have a dramatic effect on the prediction (especially for regression problems), I choosed to manage them.

I used the Tukey method (Tukey JW., 1977) to detect outliers which defines an interquartile range comprised between the 1st and 3rd quartile of the distribution values (IQR). An outlier is a row that have a feature value outside the (IQR +- an outlier step).

I decided to detect outliers from the numerical 